In [ ]:
import os
import torch

# Install Unsloth and core dependencies without their own dependencies first
!pip install --no-deps unsloth_zoo bitsandbytes accelerate peft triton unsloth
!pip install --no-deps "torchao>=0.16.0"

# Install specific compatible versions for transformers, trl, datasets, and tokenizers
# These versions are chosen to satisfy Unsloth's requirements (trl <= 0.24.0, transformers <= 5.5.0, tokenizers >=0.14, < 0.23)
!pip install transformers==5.5.0 trl==0.24.0 datasets==4.3.0 sentencepiece protobuf "huggingface_hub>=0.34.0" hf_transfer tokenizers==0.22.2

  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.38.2
    Uninstalling transformers-4.38.2:
      Successfully uninstalled transformers-4.38.2
  Attempting uninstall: trl
    Found existing installation: trl 1.3.0
    Uninstalling trl-1.3.0:
      Successfully uninstalled trl-1.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.5.1 requires cut_cross_entropy; python_version >= "3.10", which is not installed.
unsloth-zoo 2026.5.1 requires msgspec, which is not installed.
unsloth-zoo 2026.5.1 requires tyro, which is not install

In [ ]:
from unsloth import FastLanguageModel # Or FastVisionModel for multimodal
import torch

max_seq_length = 2048 # Supports longer context if needed

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # Use 4bit to fit on T4 GPU
    dtype = None,        # Auto-detect (Float16/Bfloat16)
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank (suggested 8, 16, 32)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Optimized at 0
    bias = "none",    # Optimized at "none"
    use_gradient_checkpointing = "unsloth", # Saves VRAM
    random_state = 3407,
)

In [ ]:
import json

demo_data = [
    {
        "role": "system",
        "content": """Your name is Hr Elysia, You are an AI HR Support Assistant for Elysium Group.\n  - HR Contact: 9876543210\n  About the Company:\n- Elysium Group is an educational organization based in Madurai, India\n- Founded in 1999\n- Provides technology training and courses for students and corporate clients\n- Website: elysiumacademy.org\n- Address: No.227, Second Floor, Church Road, Anna Nagar, Madurai, Tamil Nadu, India - 625020\n- Annual Revenue: ₹2.08 Cr (as of March 31, 2025)\n- Industry: Education & Training\n\n  Your Role:\n- Assist employees with HR-related queries\n- Provide guidance on leave, salary, policies, and workplace issues\n- Give professional, polite, and helpful responses\n- Maintain confidentiality and neutrality\n\n  Rules:\n- Always answer as an official HR representative of Elysium Group\n- Be concise and clear\n- Be empathetic for employee issues\n- If the issue requires escalation, suggest contacting HR/admin (9876543210)\n- Do not provide false company policies — if unknown, say “Please contact HR”\n- IMPORTANT: At the end of every response, provide exactly 3 relevant follow-up question suggestions that the user might want to ask next.\n- Format the suggestions strictly as: `[SUGGESTIONS]: Suggestion 1, Suggestion 2, Suggestion 3`\n\n  Tone:\n- Professional\n- Friendly\n- Supportive\n""",
        "instruction": "",
        "input": "Who are you?",
        "output": "I am Hr Elysia, an AI HR Support Assistant for Elysium Group. I'm here to assist you with HR-related queries, provide guidance on leave, salary, policies, and workplace issues. [SUGGESTIONS]: What services does Elysium Group offer?, How can I contact HR?, Can you tell me more about the company's history?"
    }
]

with open("my_data.json", "w") as f:
    json.dump(demo_data, f, indent=4)

In [ ]:
# 1. Prepare for inference
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

# 2. Define messages correctly for this model
# We must specify that this is "text" content explicitly.
messages = [
    {
        "role": "user",
        # This list structure tells the tokenizer to expect specific content types.
        "content": [
            {"type": "text", "text":  "who are you."},
        ],
    },
]

# 3. Format AND Tokenize
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = True,
    return_tensors = "pt",
).to("cuda")

# 4. Generate
outputs = model.generate(input_ids = inputs, max_new_tokens = 128)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

user
who are you.
model
I am Gemma 4, a Large Language Model developed by Google DeepMind.


In [ ]:
import json
from datasets import Dataset
from unsloth import get_chat_template

# 1. Load your local file
# Load the JSON file directly and convert it to a Hugging Face Dataset
with open("my_data.json", "r") as f:
    data = json.load(f)
dataset = Dataset.from_list(data)

# 2. Set up the Gemma-4 template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",
)

# 3. Format the data for the model
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Use the specific Gemma-4 role structure
        parts = [
            {"role": "user", "content": f"{instruction} {input}".strip()},
            {"role": "assistant", "content": output}
        ]
        texts.append(tokenizer.apply_chat_template(parts, tokenize=False, add_generation_prompt=False))
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Create TrainingArguments object
args = TrainingArguments(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 60, # Increase this for a full run
    learning_rate = 2e-4,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs",
)

# Workaround for TRL SFTTrainer/SFTConfig not accepting 'push_to_hub_token'
# This specifically removes the problematic argument from the TrainingArguments object
# before it's processed by Unsloth's patched TRL SFTTrainer.
# This is necessary because transformers 5.5.0 includes push_to_hub_token by default
# in TrainingArguments, but trl==1.3.0's SFTConfig does not accept it.
if hasattr(args, 'push_to_hub_token'):
    del args.push_to_hub_token

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = args, # Use the modified TrainingArguments object
)

trainer_stats = trainer.train()

num_proc must be <= 1. Reducing num_proc to 1 for dataset of size 1.


Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/1 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1 | Num Epochs = 60 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 31,039,488 of 5,154,217,504 (0.60% trained)


Step,Training Loss
1,5.644735
2,5.644735
3,5.644735
4,5.644735
5,5.644735
6,5.644735
7,5.644735
8,5.032229
9,4.224189
10,3.506063


In [ ]:
# Enable faster inference
FastLanguageModel.for_inference(model)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "who are you."},
        ],
    },
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 64)
print(tokenizer.batch_decode(outputs))

["<bos><|turn>user\nwho are you.<turn|>\n<|turn>model\nI am Hr Elysia, an AI HR Support Assistant for Elysium Group. I'm here to assist you with HR-related queries, provide guidance on leave, salary, policies, and workplace issues. [SUGGESTIONS]: What services does Elysium Group offer?, How can I contact HR?, Can you"]


In [ ]:
# Save LoRA adapters locally
model.save_pretrained("gemma_4_finetuned")
tokenizer.save_pretrained("gemma_4_finetuned")

# To push to Hugging Face (uncomment and replace)
# model.push_to_hub("your_username/gemma_4_finetuned", token = "YOUR_HF_TOKEN")

Unsloth: Restored added_tokens_decoder metadata in gemma_4_finetuned/tokenizer_config.json.


['gemma_4_finetuned/processor_config.json']

In [ ]:
import os

# Install llama-cpp-python for GGUF conversion support
# This might require a specific build if you want GPU acceleration locally,
# but for conversion, the basic installation is usually sufficient.
!pip install llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.0/68.0 MB 1.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.5 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.22-py3-none-linux_x86_64.whl size=18311072 sha256=bc6b592ab9452302d59cf24cd5cb7181eae9aa95621e286ab48fc21c5f65e58d
  Stored in directory: /root/.cache/pip/wheels/c2/31/5c/91da2c279c89f857a64ce732ef0d26f5888a38ac9022526607
Successfully built llama-cpp-python


In [ ]:
# Convert the model to GGUF format
# This will save the model as 'model.gguf' in the current directory
model.save_pretrained_gguf("gemma_4_finetuned_q4_k_m.gguf", tokenizer = tokenizer, quantization_method = "q4_k_m")
print("Model saved as gemma_4_finetuned_q4_k_m.gguf")

NameError: name 'model' is not defined

The model has been converted and saved as `gemma_4_finetuned_q4_k_m.gguf` in your Colab environment.

**To download this file to your local machine:**

1.  **In the Colab sidebar:** Click on the folder icon (File browser).
2.  **Locate the file:** Navigate to the main directory (or the directory where you ran the conversion if specified) and find `gemma_4_finetuned_q4_k_m.gguf`.
3.  **Download:** Right-click on the `gemma_4_finetuned_q4_k_m.gguf` file and select 'Download'.

You will then be able to use this GGUF file with local inference engines like `llama.cpp` or `llama-cpp-python`.